<a href="https://colab.research.google.com/github/Sakinat-Folorunso/OOU_CSC309_Artificial_Intelligence/blob/main/notebooks/CSC309_Week05_KRR_Logic_Student_Centred.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CSC 309 – Artificial Intelligence
**Week 5 Practical (PH):** Knowledge representation & reasoning — predicate logic, clause form, resolution

**Instructor:** Dr Sakinat Folorunso · Associate Professor of AI Systems · Department of Computer Sciences, Olabisi Onabanjo University (OOU), Ago-Iwoye, Ogun State, Nigeria  
**Course:** CSC 309 (2 units; LH 15, PH 45) · **Mode:** student-centred, hands-on, Google Colab  
**CCMAS learning outcomes covered:** LO6, LO7

> 🎓 **Instructor note** – this notebook is the *STUDENT (TODO cells are stubs)* version.


## How to use this notebook
1. Fill in the **Group Log**, then do the **Do Now** with your partner.
2. Run the **Setup** cell once (*Runtime → Run all* is safe: nothing here crashes if a TODO is still empty).
3. Work top to bottom. Only edit cells marked **`# TODO(Student)`**.
4. Each task ends with an **auto-check** (✅/❌). Keep going until it is ✅.
5. Finish with the **Reflection**. Fast finishers: try the **Extensions**.

6. **Read the comments.** Every line of code has a plain-English comment above it. Read the comment first, predict what the line does, then run the cell and check your prediction.
7. Hand in the **Assignment** (marks and deadline are in the notebook) after the lab.

*Every random draw uses `SEED = 42`, so your numbers should match your classmates' numbers.*


## Lab tasks at a glance

| Part | Lab task(s) you must complete |
|---|---|
| Part A — A knowledge base you can query | Task A1 — Write two more queries |
| Part B — From English to clause form | Task B1 — Translate English → predicate logic (quantifier scope!) |
| Part C — Resolution refutation: who killed Tuna? | *Guided lab: run every cell, read the comments, explain the output in your own words* |
| Part D — Non-monotonic reasoning (defaults) | *Guided lab: run every cell, read the comments, explain the output in your own words* |
| Part E — Probabilistic reasoning (when facts are uncertain) | *Guided lab: run every cell, read the comments, explain the output in your own words* |
| **Assignment** | see **📝 Assignment / deliverables** near the end |
| **Reflection** | short written reflection at the end (5–7 sentences) |
| *Extensions* | optional, for fast finishers |

In [ ]:
#@title 🧑🏽‍🤝‍🧑🏾 Group Log (fill before you start)
# Form field: type your group members' names here
group_members = "Type names here"   #@param {type:"string"}
# Form field: assign roles (driver, navigator, recorder, presenter)
roles_notes   = "Driver / Navigator / Recorder / Presenter"  #@param {type:"string"}
# Print them so your notebook records who did what
print("👥 Group:", group_members); print("📝 Roles:", roles_notes)


In [ ]:
#@title 🔧 Setup (run once) — seed, logging, helpers
# Standard-library modules used by every lab (os, re, math, random, logging, time ...)
import sys, os, re, math, random, logging, subprocess, warnings, time, itertools
# NumPy: fast arrays and random numbers
import numpy as np
# One seed for the whole course so every run gives the same numbers
SEED = 42                                    # one seed for the whole course → reproducible results
# Seed both Python's and NumPy's random generators
random.seed(SEED); np.random.seed(SEED)
# Configure logging so messages print as "LEVEL | message"
logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s", force=True)
# A named logger for this course
log = logging.getLogger("csc309")
# Hide noisy library warnings so output stays readable
warnings.filterwarnings("ignore")

def ensure(pip_name, import_name=None):
    """Import a package; pip-install it first if it is missing (Colab-friendly)."""
    # Try to import the library
    try:
        # Return the imported module
        return __import__(import_name or pip_name)
    # If it is not installed ...
    except ImportError:
        # ... tell the student we are installing it
        log.info("installing %s ...", pip_name)
        # ... and install it with pip
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])
        # Return the imported module
        return __import__(import_name or pip_name)

# List that stores every check result
_CHECKS = []
def check(name, condition, hint=""):
    """Auto-check. Never raises, so 'Run all' always finishes. Pass a lambda to catch your own bugs."""
    # Try to import the library
    try:
        # Run the check: call it if it is a function, otherwise use the value
        ok = bool(condition() if callable(condition) else condition)
        # Default: no error message
        err = ""
    # If the student's code crashes, we report instead of stopping the notebook
    except Exception as e:                    # student code may crash; report instead of stopping
        # Mark as failed and keep the error message
        ok, err = False, f"{type(e).__name__}: {e}"
    # Store the result
    _CHECKS.append((name, ok))
    # Print a ✅ or ❌ line with a hint on failure
    print(("✅ PASS       " if ok else "❌ TRY AGAIN  ") + name + ("" if ok else f"   → {err or hint}"))
    # Return whether the check passed
    return ok

# Print how many checks passed
def checks_summary():
    # Count the passed checks
    passed = sum(ok for _, ok in _CHECKS)
    # Print the total
    print(f"\nChecks passed: {passed}/{len(_CHECKS)}")

def mean_ci(x, conf=0.95):
    """Mean, sample std and a two-sided t-based confidence interval."""
    # SciPy for the t-distribution
    from scipy import stats
    # Convert to a float array and count the observations
    x = np.asarray(x, dtype=float); n = len(x)
    # Mean and sample standard deviation (n-1 divisor)
    m = x.mean(); s = x.std(ddof=1) if n > 1 else 0.0
    # Half-width of the 95% confidence interval from the t-distribution
    h = stats.t.ppf((1 + conf) / 2, n - 1) * s / math.sqrt(n) if n > 1 else 0.0
    # Return mean, SD, lower and upper confidence limits
    return m, s, m - h, m + h

# Print a one-line statistical summary
def report(label, x):
    # Compute the statistics
    m, s, lo, hi = mean_ci(x)
    # Print mean, SD, CI and n
    print(f"{label:<28} mean={m:8.2f}  sd={s:7.2f}  95% CI=[{lo:.2f}, {hi:.2f}]  (n={len(x)})")

def paired_wilcoxon(a, b):
    """Two-sided Wilcoxon signed-rank test for paired samples (same seeds/instances)."""
    # SciPy for the t-distribution
    from scipy import stats
    # Convert both samples to arrays
    a, b = np.asarray(a, float), np.asarray(b, float)
    # If the two samples are identical the test is undefined
    if np.allclose(a, b):
        # Report p = 1 (no difference)
        return 1.0
    # Paired Wilcoxon signed-rank test p-value
    return float(stats.wilcoxon(a, b).pvalue)

# Setup finished
print("✅ Setup complete — seed =", SEED)


### Learning objectives (with Bloom level)
| # | By the end of this lab you can… | Bloom | CCMAS LO |
|---|---|---|---|
| 1 | represent facts and queries in a knowledge base and answer them by **unification** | Apply | LO6 |
| 2 | **translate English sentences into predicate logic** (quantifier scope matters!) | Apply | LO7 |
| 3 | convert a formula into **clause form** (eliminate →, NNF, Skolemise, drop ∀, CNF) | Apply | LO7 |
| 4 | use **resolution refutation** to answer a query from a set of statements | Apply / Analyse | LO7 |
| 5 | contrast **monotonic** and **non-monotonic** reasoning, and reason **probabilistically** | Analyse | LO6 |

**Key ideas.** A knowledge-representation (KR) language needs a *syntax*, a *semantics* and an *inference procedure*. First-order logic (FOL) has terms (constants, variables, functions), predicates, connectives $\neg,\wedge,\vee,\rightarrow,\leftrightarrow$ and quantifiers $\forall,\exists$.

**Resolution rule (propositional).** From clauses $(\ell \vee A)$ and $(\neg\ell \vee B)$ infer $(A \vee B)$. In FOL, literals $\ell_1$ and $\neg\ell_2$ resolve when they **unify** with a most general unifier $\theta$: the resolvent is $(A\vee B)\theta$.

**Refutation.** To show $KB \models \alpha$, add $\neg\alpha$ to the clauses of $KB$ and derive the **empty clause** $\square$. Resolution is *sound* and *refutation-complete* for FOL.

**Do Now (3 min).** Write in English two different meanings of *"Everybody loves somebody."* How would you tell them apart in logic? (You will test your answer in Task B1.)

## Part A — A knowledge base you can query
Facts are tuples `('Parent','Ada','Sam')`. A **query** is a pattern with variables (lowercase). `ask_all` finds every binding that satisfies a *conjunction* of goals, using **unification**.

In [ ]:
# itertools for combinations
import itertools
# deque for queues
from collections import deque

In [ ]:
#@title 🧮 A small first-order logic engine: formula → clause form → unification → resolution
# Representation (plain Python tuples):
#   variables  = lowercase strings      'x', 'y'            constants = Capitalised strings   'Ada', 'Tuna'
#   atoms      = ('Pred', t1, t2, …)     e.g. ('Loves', 'x', 'Ada')
#   formulas   = ('not',F) ('and',F,G) ('or',F,G) ('implies',F,G) ('iff',F,G) ('forall','x',F) ('exists','x',F)
# Words that are logical operators rather than predicate names
CONNECTIVES = {"not", "and", "or", "implies", "iff", "forall", "exists"}
# A term is a variable if it is a string starting with a lowercase letter
is_var = lambda t: isinstance(t, str) and t[:1].islower()
# Counter that produces x1, x2, ... so every new variable is unique
_fresh = itertools.count(1)

def show(t):
    """Pretty-print a term, atom, formula, literal or clause."""
    # A plain symbol prints as itself
    if isinstance(t, str): return t
    # A clause (set of literals) prints with braces
    if isinstance(t, frozenset):
        # Join the sorted literals; an empty clause is the contradiction square
        return "{" + ", ".join(sorted(show(l) for l in t)) + "}" if t else "□ (empty clause)"
    # A literal is a (sign, atom) pair, e.g. (False, "P") means not-P
    if isinstance(t, tuple) and len(t) == 2 and isinstance(t[0], bool):      # literal (sign, atom)
        # Put a negation sign in front when the sign is False
        return ("" if t[0] else "¬") + show(t[1])
    # Otherwise it is a tuple whose first item is the operator
    op = t[0]
    # not F is printed as ¬F
    if op == "not":     return "¬" + show(t[1])
    # Binary connectives are printed infix
    if op in ("and", "or", "implies", "iff"):
        # Choose the printed symbol for the connective
        sym = {"and": " ∧ ", "or": " ∨ ", "implies": " → ", "iff": " ↔ "}[op]
        # Wrap in brackets: (left symbol right)
        return "(" + show(t[1]) + sym + show(t[2]) + ")"
    # Quantifiers print as ∀x body or ∃x body
    if op in ("forall", "exists"): return ("∀" if op == "forall" else "∃") + t[1] + " " + show(t[2])
    # Anything else is a predicate or function: name(arg, arg)
    return op + "(" + ", ".join(show(a) for a in t[1:]) + ")"

# ---------- 1. formula → clause form --------------------------------------------------------
# Step 1 of CNF: remove -> and <-> using their definitions
def elim_arrows(f):
    # The first item of the tuple names the operator
    op = f[0]
    # A -> B is the same as (not A) or B
    if op == "implies": return ("or", ("not", elim_arrows(f[1])), elim_arrows(f[2]))
    # A <-> B means both directions hold
    if op == "iff":
        # Eliminate arrows inside both sides first
        a, b = elim_arrows(f[1]), elim_arrows(f[2])
        # (not A or B) and (not B or A)
        return ("and", ("or", ("not", a), b), ("or", ("not", b), a))
    # and/or: keep the operator, clean both children
    if op in ("and", "or"):        return (op, elim_arrows(f[1]), elim_arrows(f[2]))
    # not: clean the child
    if op == "not":                return ("not", elim_arrows(f[1]))
    # Quantifiers: keep the variable, clean the body
    if op in ("forall", "exists"): return (op, f[1], elim_arrows(f[2]))
    # An atom has no arrows, so return it unchanged
    return f                                                    # atom

def nnf(f):
    """Negation normal form: push ¬ inwards (De Morgan, quantifier duality)."""
    # The first item of the tuple names the operator
    op = f[0]
    # Step 2: push negations inward until they sit on atoms (NNF)
    if op == "not":
        # g is the thing being negated; gop is its operator
        g = f[1]; gop = g[0]
        # Double negation cancels: not not F = F
        if gop == "not":    return nnf(g[1])
        # De Morgan: not(A and B) = not A or not B
        if gop == "and":    return ("or",  nnf(("not", g[1])), nnf(("not", g[2])))
        # De Morgan: not(A or B) = not A and not B
        if gop == "or":     return ("and", nnf(("not", g[1])), nnf(("not", g[2])))
        # not forall x F = exists x not F
        if gop == "forall": return ("exists", g[1], nnf(("not", g[2])))
        # not exists x F = forall x not F
        if gop == "exists": return ("forall", g[1], nnf(("not", g[2])))
        # A negated atom is already in normal form
        return f                                                # ¬atom
    # and/or: recurse into both sides
    if op in ("and", "or"):        return (op, nnf(f[1]), nnf(f[2]))
    # Quantifiers: recurse into the body
    if op in ("forall", "exists"): return (op, f[1], nnf(f[2]))
    # Atoms are returned unchanged
    return f

def subst(t, env):
    """Apply a substitution env (var → term) once, to a term/atom."""
    # A symbol: replace it if it is a variable that has a binding in env
    if isinstance(t, str): return env.get(t, t) if is_var(t) else t
    # A compound term: substitute inside every argument
    return (t[0],) + tuple(subst(a, env) for a in t[1:])

def skolemize(f, env=None, univ=()):
    """Standardise variables apart, replace ∃ by Skolem constants/functions, drop ∀ (NNF input)."""
    # env maps variable -> replacement; op is the operator
    env = env or {}; op = f[0]
    # and/or: skolemize both sides with the same bindings
    if op in ("and", "or"): return (op, skolemize(f[1], env, univ), skolemize(f[2], env, univ))
    # forall x: rename x to a fresh variable and remember it in univ
    if op == "forall":
        # Make a brand-new variable name
        v = f"x{next(_fresh)}"
        # Continue on the body with x bound to the new name
        return skolemize(f[2], {**env, f[1]: v}, univ + (v,))
    # exists x: replace x by a Skolem constant or function
    if op == "exists":
        # Fresh index so each Skolem symbol is unique
        k = next(_fresh)
        # If universal variables are in scope the Skolem term is a function of them; otherwise a constant
        term = (f"sk{k}",) + univ if univ else f"Sk{k}"          # Skolem function of the ∀ variables in scope
        # Continue on the body with x replaced by the Skolem term
        return skolemize(f[2], {**env, f[1]: term}, univ)
    # not (applied to an atom by now): recurse inside
    if op == "not": return ("not", skolemize(f[1], env, univ))
    # An atom: apply the substitution to its arguments
    return subst(f, env)

def to_cnf(f):
    """Quantifier-free NNF → list of clauses; clause = frozenset of literals (sign, atom)."""
    # The first item of the tuple names the operator
    op = f[0]
    # and: the clause lists of both sides are simply joined
    if op == "and": return to_cnf(f[1]) + to_cnf(f[2])
    # or: distribute, combining every clause of the left with every clause of the right
    if op == "or":  return [a | b for a in to_cnf(f[1]) for b in to_cnf(f[2])]
    # not atom becomes the one-literal clause {¬atom}
    if op == "not": return [frozenset([(False, f[1])])]
    # An atom becomes the one-literal clause {atom}
    return [frozenset([(True, f)])]

def to_clauses(f):
    """Full pipeline: eliminate →/↔, NNF, Skolemise, drop ∀, distribute ∨ over ∧; drop tautologies."""
    # Full pipeline: remove arrows, push negations, skolemize, convert to clauses
    cl = to_cnf(skolemize(nnf(elim_arrows(f))))
    # out will hold the cleaned clauses
    out = []
    # Look at each clause
    for c in cl:
        # Skip tautologies: a clause containing both P and ¬P is always true
        if any((not s, a) in c for s, a in c): continue          # tautology  P ∨ ¬P
        # Keep a clause only once (drop duplicates)
        if c not in out: out.append(c)
    # Hand back all legal moves
    return out

# ---------- 2. unification ---------------------------------------------------------------
# walk follows variable bindings in s until it reaches a non-bound term
def walk(t, s):
    # While t is a bound variable, replace it by its binding
    while isinstance(t, str) and is_var(t) and t in s: t = s[t]
    # Return the fully dereferenced term
    return t
# Occurs check: does variable v appear inside term t?
def occurs(v, t, s):
    # Dereference t first
    t = walk(t, s)
    # True if t is v itself or v occurs in any argument of a compound term
    return t == v or (isinstance(t, tuple) and any(occurs(v, a, s) for a in t[1:]))
def unify(x, y, s=None):
    """Most general unifier of terms x, y (dict) or None. Includes the occurs check."""
    # Start from an empty substitution if none was given
    s = {} if s is None else s
    # Dereference both terms so we compare the current values
    x, y = walk(x, s), walk(y, s)
    # Identical terms already unify; return the substitution unchanged
    if x == y: return s
    # x is a variable: bind it to y (unless the occurs check fails)
    if isinstance(x, str) and is_var(x): return None if occurs(x, y, s) else {**s, x: y}
    # y is a variable: bind it to x (unless the occurs check fails)
    if isinstance(y, str) and is_var(y): return None if occurs(y, x, s) else {**s, y: x}
    # Two compound terms unify only if functor and arity match
    if isinstance(x, tuple) and isinstance(y, tuple) and x[0] == y[0] and len(x) == len(y):
        # Unify the arguments pairwise
        for a, b in zip(x[1:], y[1:]):
            # Extend the substitution with each pair
            s = unify(a, b, s)
            # If any pair fails, the whole unification fails
            if s is None: return None
        # Return the scrambled board
        return s
    # Nothing matched: unification fails (None)
    return None
# Apply the final substitution all the way through a term
def resolve_term(t, s):
    # Dereference t first
    t = walk(t, s)
    # Constants unchanged; compound terms are resolved argument by argument
    return t if isinstance(t, str) else (t[0],) + tuple(resolve_term(a, s) for a in t[1:])

# ---------- 3. resolution refutation -------------------------------------------------------
# Counter for renaming variables apart
_rename_id = itertools.count(1)
# Give a clause fresh variable names so two clauses never share variables
def rename_apart(c):
    # Next unused number
    n = next(_rename_id)
    # Helper that renames one term
    def r(t):
        # Variables get a suffix _n; constants stay the same
        if isinstance(t, str): return f"{t}_{n}" if is_var(t) else t
        # Compound terms: rename each argument
        return (t[0],) + tuple(r(a) for a in t[1:])
    # Rebuild the clause with renamed atoms
    return frozenset((s, r(a)) for s, a in c)

def canon(c):
    """Canonical form of a clause (variables renamed by first occurrence) used for duplicate detection."""
    # Canonical-name table: first variable seen becomes v0, next v1, ...
    names = {}
    # Helper that renames one term
    def r(t):
        # A symbol
        if isinstance(t, str):
            # Name variables by order of appearance so equivalent clauses look identical
            if is_var(t): names.setdefault(t, f"v{len(names)}"); return names[t]
            # Return the fully dereferenced term
            return t
        # Compound terms: rename each argument
        return (t[0],) + tuple(r(a) for a in t[1:])
    # Canonical form of a clause: rename variables and sort literals
    return frozenset((s, r(a)) for s, a in sorted(c, key=lambda l: (show(l))))

# All clauses obtainable by resolving c1 with c2
def resolvents(c1, c2):
    # Rename apart first; out collects new clauses
    c1, c2 = rename_apart(c1), rename_apart(c2); out = []
    # Try every literal of c1 ...
    for (s1, a1) in c1:
        # ... against every literal of c2
        for (s2, a2) in c2:
            # Only literals of opposite sign can be resolved
            if s1 != s2:
                # Try to unify the two atoms (find a substitution u)
                u = unify(a1, a2)
                # If they unify, build the resolvent
                if u is not None:
                    # Remove the two resolved literals, join what is left, apply u
                    new = {(s, resolve_term(a, u)) for (s, a) in (c1 - {(s1, a1)}) | (c2 - {(s2, a2)})}
                    # Add the new clause to the list of results
                    out.append(frozenset(new))
    # Hand back all legal moves
    return out

def refute(clauses, max_clauses=3000, sos=None):
    """Resolution refutation (set-of-support, breadth-first). Returns the proof (list of steps) or None."""
    # kb = all clauses; sos = "set of support", the clauses we are allowed to start from
    kb = [c for c in clauses]; sos = list(sos) if sos is not None else list(clauses)
    # Table of every clause found: key -> (clause, rule used, parent keys)
    nodes = {canon(c): (c, "given", ()) for c in kb}                 # canon → (clause, rule, parent keys)
    # Queue of clauses still to be combined (breadth-first)
    queue = deque(canon(c) for c in sos)
    # Stop when the queue empties or too many clauses have been generated
    while queue and len(nodes) < max_clauses:
        # Take the next clause to process
        k1 = queue.popleft(); c1 = nodes[k1][0]
        # Combine it with every clause found so far
        for k2 in list(nodes):
            # Each resolvent of the pair
            for r in resolvents(c1, nodes[k2][0]):
                # Canonical key so repeated clauses are recognised
                kr = canon(r)
                # Skip clauses we already have
                if kr in nodes: continue
                # Store the new clause with its parents and queue it
                nodes[kr] = (r, "resolve", (k1, k2)); queue.append(kr)
                # The empty clause means contradiction, so the query is proved
                if not r:                                             # empty clause ⇒ contradiction
                    # Return the proof tree
                    return _proof(nodes, kr)
    # Nothing matched: unification fails (None)
    return None

# Turn the table of clauses into a numbered proof listing
def _proof(nodes, goal):
    # order = clauses in proof order; seen = visited keys
    order, seen = [], set()
    # Depth-first visit so parents are listed before children
    def visit(k):
        # Skip anything already listed
        if k in seen: return
        # Mark as seen
        seen.add(k)
        # Visit the parents first
        for p in nodes[k][2]: visit(p)
        # Then add this clause
        order.append(k)
    # Start from the empty clause and walk back through its ancestors
    visit(goal)
    # Number the proof lines 1, 2, 3...
    idx = {k: i + 1 for i, k in enumerate(order)}
    # Return (line number, clause, rule, parent line numbers) tuples
    return [(idx[k], nodes[k][0], nodes[k][1], tuple(idx[p] for p in nodes[k][2])) for k in order]

# Print a proof in readable form
def print_proof(proof):
    # Each proof line
    for i, c, rule, parents in proof:
        # Show line number, the clause, and which rule produced it
        print(f"{i:>3}. {show(canon(c)):<48} {'given' if rule == 'given' else 'resolve(' + ', '.join(map(str, parents)) + ')'}")

def prove(kb_formulas, query):
    """KB ⊨ query ?  Refute KB ∧ ¬query. Returns the proof or None (no refutation found)."""
    # Convert every knowledge-base formula to clauses
    kb = [c for f in kb_formulas for c in to_clauses(f)]
    # Negate the query and convert it: proof by contradiction
    neg = to_clauses(("not", query))
    # Run resolution on KB plus negated query
    return refute(kb + neg, sos=neg)

### 🔍 Code walkthrough — answering a query by unification
A query like *"Ada's grandchildren"* is two goals: `Parent(Ada, y)` and `Parent(y, z)`. `ask_all` takes the first goal, tries to **unify** it with every fact, and for each success carries the new bindings forward to the remaining goals. When no goals are left, the bindings are one complete answer. This is exactly how Prolog works, in 10 lines.

In [ ]:
#@title 👪 Facts + conjunctive queries
# The knowledge base: facts as (Predicate, arg1, arg2) tuples
FACTS = [("Parent","Ada","Sam"), ("Parent","Sam","Tayo"), ("Parent","Sam","Ife"), ("Parent","Ife","Bola"),
         ("Female","Ada"), ("Female","Ife"), ("Male","Sam"), ("Male","Tayo"), ("Female","Bola")]

def ask_all(facts, goals, s=None):
    """All substitutions satisfying every goal in `goals` (a join by unification)."""
    # Start from an empty substitution
    s = s or {}
    # No goals left: return the current substitution as one answer
    if not goals: return [s]
    # Collect the answers
    out = []
    # Try every fact for the first goal
    for fact in facts:
        # Unify the goal with the fact
        s2 = unify(goals[0], fact, s)
        # If it matches, solve the remaining goals with the extended substitution
        if s2 is not None: out += ask_all(facts, goals[1:], s2)
    # Return all answers
    return out

# Ask: who are Ada's grandchildren? (y and z are variables)
grandkids = ask_all(FACTS, [("Parent","Ada","y"), ("Parent","y","z")])
# Resolve z in each answer and print the sorted set
print("Ada's grandchildren:", sorted({resolve_term("z", s) for s in grandkids}))

### Task A1 — Write two more queries
1. `Q_AUNT_OR_UNCLE`-style: **Who are Sam's grandchildren who are female?** (goals list `Q1`).
2. **Who are siblings?** (goals list `Q2` — two different children of the same parent; remember a person is not their own sibling, filter afterwards).

In [ ]:
# TODO(Student): write the goal lists
# TODO: write a query for Sam's granddaughters
Q1 = []      # e.g. [("Parent","Sam","y"), ...]
# TODO: write a query for sibling pairs
Q2 = []
# TODO: collect sibling pairs here
sib_pairs = set()    # fill with {(child1, child2), ...} computed from ask_all(FACTS, Q2)

In [ ]:
#@title ✅ Auto-check for Task A1
check("Sam's female grandchild is Bola only", lambda: {resolve_term("z", s) for s in ask_all(FACTS, Q1)} == {"Bola"})
check("siblings: Tayo & Ife (both directions)", lambda: sib_pairs == {("Tayo","Ife"), ("Ife","Tayo")})

### Unification and the occurs check (class note §5.4)
Unifying $\mathrm{Teaches}(x,f(x))$ with $\mathrm{Teaches}(\mathrm{Ada},z)$ gives $\{x/\mathrm{Ada},\ z/f(\mathrm{Ada})\}$. A variable may **not** be bound to a term that contains it (the *occurs check*): $x/f(x)$ would build an infinite term, so `unify` must fail. Run the cell and read each result aloud before the check.

In [ ]:
#@title 🔗 Unification demo + auto-check
# Unify Teaches(x, f(x)) with Teaches(Ada, z)
u1 = unify(("Teaches", "x", ("f", "x")), ("Teaches", "Ada", "z"))
# Show the most general unifier
print("mgu:", {v: resolve_term(v, u1) for v in u1})                       # x → Ada, z → f(Ada)
# Occurs check: x cannot equal f(x), so the result is None
u2 = unify("x", ("f", "x"))                                               # occurs check → None
# Print the failed unification
print("unify(x, f(x)) =", u2)
# Unify P(x, Ada) with P(Bola, y)
u3 = unify(("P", "x", "Ada"), ("P", "Bola", "y"))
# Show the substitution
print("unify(P(x,Ada), P(Bola,y)) =", {v: resolve_term(v, u3) for v in u3})
check("note Example 5.4: x/Ada and z/f(Ada)", lambda: resolve_term("x", u1) == "Ada" and resolve_term("z", u1) == ("f", "Ada"))
check("occurs check rejects x / f(x)",       lambda: u2 is None)
check("different constants do not unify",    lambda: unify(("P", "Ada"), ("P", "Bola")) is None)

## Part B — From English to clause form
### The pipeline (watch every stage)
1. Eliminate $\rightarrow$ and $\leftrightarrow$. 2. Move $\neg$ inwards (NNF). 3. Standardise variables apart. 4. **Skolemise**: replace each $\exists$ by a fresh constant (or a function of the enclosing $\forall$ variables). 5. Drop $\forall$. 6. Distribute $\vee$ over $\wedge$ → a set of **clauses**.

In [ ]:
#@title 🔬 Step-by-step clause conversion
# Show the six steps of CNF conversion
def explain_clause_form(f):
    # Print the original formula
    print("Original   :", show(f))
    # Step 1: remove implications
    s1 = elim_arrows(f);  print("1. no →,↔  :", show(s1))
    # Step 2: push negations in
    s2 = nnf(s1);         print("2. NNF     :", show(s2))
    # Steps 3-5: Skolemise and drop quantifiers
    s3 = skolemize(s2);   print("3–5. Skolemise, drop ∀ :", show(s3))
    # Step 6: print the final clauses
    print("6. clauses :", " ∧ ".join(show(c) for c in to_clauses(f)), "\n")

# Example: every human is mortal
explain_clause_form(("forall","x",("implies",("Human","x"),("Mortal","x"))))
# Example: a harder one with a nested quantifier
explain_clause_form(("forall","x",("implies",("exists","y",("and",("Dog","y"),("Owns","x","y"))),("AnimalLover","x"))))

### Task B1 — Translate English → predicate logic (quantifier scope!)
Write each sentence as a nested tuple (see the examples above). Use predicates `Student, Attends, Passes, Loves` and constant `Ada`.

| | English | Intended logic |
|---|---|---|
| `T1` | Every student who attends lectures passes. | $\forall x\,(\text{Student}(x)\wedge\text{Attends}(x)\rightarrow\text{Passes}(x))$ |
| `T2` | Ada is a student who attends lectures. | $\text{Student}(Ada)\wedge\text{Attends}(Ada)$ |
| `T3` | Everybody loves somebody. | $\forall x\,\exists y\,\text{Loves}(x,y)$ |
| `T4` | There is somebody whom everybody loves. | $\exists y\,\forall x\,\text{Loves}(x,y)$ |

In [ ]:
# TODO(Student): replace None with tuple formulas
# TODO: translate sentence 1 into logic
T1 = None
# TODO: translate sentence 2
T2 = None
# TODO: translate sentence 3
T3 = None
# TODO: translate sentence 4
T4 = None

In [ ]:
#@title ✅ Auto-check for Task B1 (we test *meaning* with the prover, not spelling)
check("T1,T2 ⊨ Passes(Ada)",                lambda: prove([T1, T2], ("Passes","Ada")) is not None)
check("T1,T2 ⊭ Passes(Bola)",               lambda: prove([T1, T2], ("Passes","Bola")) is None)
check("T3 needs a Skolem FUNCTION of x",     lambda: any(isinstance(a, tuple) for c in to_clauses(T3) for _, at in c for a in at[1:]))
check("T4 needs a Skolem CONSTANT",          lambda: any(isinstance(a, str) and a.startswith("Sk") for c in to_clauses(T4) for _, at in c for a in at[1:]))
check("T4 ⊨ T3  (someone loved by all ⇒ everyone loves someone)", lambda: prove([T4], T3) is not None)
check("T3 ⊭ T4  (the converse does NOT hold)",                    lambda: prove([T3], T4) is None)

**Discuss.** Why does `T3 ⊭ T4`? Hint: look at the clause forms `Loves(x, sk(x))` and `Loves(x, Sk)`. In the first, the loved person *depends on* $x$; in the second it is *one fixed* individual.

## Part C — Resolution refutation: who killed Tuna?
*(after Russell & Norvig)* Jack owns a dog. Every dog owner is an animal lover. No animal lover kills an animal. Either Jack or Curiosity killed the cat, Tuna. Tuna is a cat. Cats are animals. **Did Curiosity kill Tuna?**

### 🔍 Code walkthrough — proof by contradiction
We do not search for a proof of *Curiosity killed Tuna*. We **add its negation** to the knowledge base and keep resolving pairs of clauses until we derive the **empty clause** □, which is a contradiction. If the knowledge base is consistent, the contradiction can only have come from the negated goal, so the goal must be true. Read the printed proof from the bottom up.

In [ ]:
#@title 🕵️ The knowledge base and the proof
# Everyone who owns a dog is an animal lover
A = ("forall","x",("implies",("exists","y",("and",("Dog","y"),("Owns","x","y"))),("AnimalLover","x")))
# No animal lover kills an animal
B = ("forall","x",("implies",("AnimalLover","x"),("forall","y",("implies",("Animal","y"),("not",("Kills","x","y"))))))
# Jack or Curiosity killed Tuna
C = ("or",("Kills","Jack","Tuna"),("Kills","Curiosity","Tuna"))
# Tuna is a cat
D = ("Cat","Tuna")
# Cats are animals
E = ("forall","x",("implies",("Cat","x"),("Animal","x")))
# Jack owns a dog
F = ("exists","y",("and",("Dog","y"),("Owns","Jack","y")))
# The knowledge base
KB = [A, B, C, D, E, F]

# Print the clauses
print("Clause form of the KB:")
# For every formula ...
for f in KB:
    # ... show its clauses
    for c in to_clauses(f): print("  ", show(c))
# Prove that Curiosity killed Tuna
proof = prove(KB, ("Kills","Curiosity","Tuna"))
# Print whether it is entailed and show the proof
print("\nKB ⊨ Kills(Curiosity, Tuna)?", proof is not None, "\n"); print_proof(proof)
check("Curiosity killed Tuna (refutation found)", lambda: proof is not None)
check("…but Jack's guilt is NOT provable",         lambda: prove(KB, ("Kills","Jack","Tuna")) is None)
check("remove premise E (cats are animals) → proof fails", lambda: prove([A, B, C, D, F], ("Kills","Curiosity","Tuna")) is None)

**Read the proof.** Each numbered line is a clause; `resolve(i, j)` shows its parents. The first clause is the *negated query*. The last line is the **empty clause** □ — a contradiction, so the query follows from the KB.

## Part D — Non-monotonic reasoning (defaults)
Classical logic is **monotonic**: if $KB\models\alpha$ then $KB\cup\{\beta\}\models\alpha$. Humans reason by **defaults**: *birds normally fly*. Learning that Tweety is a penguin should **retract** the conclusion. We model a default as *"flies unless abnormal"* with **negation as failure** (not provable ⇒ assumed false).

In [ ]:
# TODO(Student): a bird is abnormal if it is a Penguin, an Ostrich or Injured.
# TODO: define abnormal(x)
def abnormal(x, facts):
    # Placeholder: nothing is abnormal yet
    return False

# A bird flies unless it is abnormal (default reasoning)
def flies(x, facts):
    # Birds that are not abnormal fly
    return ("Bird", x) in facts and not abnormal(x, facts)

# Known facts
facts = {("Bird","Tweety"), ("Bird","Opus"), ("Penguin","Opus"), ("Bird","Zazu")}

In [ ]:
#@title ✅ Auto-check for Part D + the classical (monotonic) contrast
# Which birds fly?
print({x: flies(x, facts) for x in ["Tweety", "Opus", "Zazu"]})
check("Tweety flies (default)",   lambda: flies("Tweety", facts) is True)
check("Opus (penguin) does not",  lambda: flies("Opus", facts) is False)
check("NON-MONOTONIC: adding Injured(Tweety) retracts 'Tweety flies'", lambda: flies("Tweety", facts | {("Injured","Tweety")}) is False)

# In classical FOL you cannot retract: adding the penguin facts makes the KB *inconsistent*.
# Rule: every bird flies
bird_flies = ("forall","x",("implies",("Bird","x"),("Flies","x")))
# Rule: penguins do not fly
penguins   = ("forall","x",("implies",("Penguin","x"),("not",("Flies","x"))))
# KB1 with only the bird rule
kb1 = [bird_flies, ("Bird","Tweety")]
# KB2 adds the penguin exception
kb2 = kb1 + [penguins, ("Penguin","Tweety"), ("Bird","Tweety")]
# KB1 entails Flies(Tweety)
print("KB1 ⊨ Flies(Tweety):", prove(kb1, ("Flies","Tweety")) is not None)
# KB2 now entails the opposite
print("KB2 ⊨ Flies(Tweety):", prove(kb2, ("Flies","Tweety")) is not None, "  and ⊨ ¬Flies(Tweety):", prove(kb2, ("not",("Flies","Tweety"))) is not None,
      " ← inconsistent: classical logic proves everything")
check("classical FOL keeps 'Flies(Tweety)' after new information (monotonic)", lambda: prove(kb2, ("Flies","Tweety")) is not None)

## Part E — Probabilistic reasoning (when facts are uncertain)
A mobile-money fraud detector raises an **alert**. Let $F$ = transaction is fraudulent, $A$ = alert. Given prior $P(F)$, sensitivity $P(A\mid F)$ and false-positive rate $P(A\mid\neg F)$, **Bayes' rule** gives
$$P(F\mid A)=\frac{P(A\mid F)\,P(F)}{P(A\mid F)\,P(F)+P(A\mid \neg F)\,P(\neg F)}.$$
Intuition first: if fraud is rare, even a good detector produces mostly **false alarms** (the *base-rate fallacy*).

In [ ]:
# TODO(Student): implement Bayes' rule
# TODO: Bayes posterior for fraud alerts
def posterior(prior, sens, fpr):
    # Placeholder
    return 0.0

In [ ]:
#@title ✅ Check + Monte-Carlo verification
# 1% base rate, 90% sensitivity, 5% false-positive rate
prior, sens, fpr = 0.01, 0.90, 0.05
# Posterior after one alert becomes the prior for the second
p1 = posterior(prior, sens, fpr); p2 = posterior(p1, sens, fpr)       # second independent alert: yesterday's posterior is today's prior
# Print both posteriors
print(f"P(fraud | 1 alert)  = {p1:.4f}\nP(fraud | 2 alerts) = {p2:.4f}")
check("P(F|A) ≈ 0.1538 (not 0.90!)", lambda: abs(p1 - 0.153846) < 1e-4)
check("two independent alerts ⇒ ≈ 0.766", lambda: abs(p2 - 0.76596) < 1e-3)

# Monte-Carlo check with a million simulated transactions
rng = np.random.default_rng(SEED); n = 1_000_000
# Each transaction is fraud with probability prior
fraud = rng.random(n) < prior
# Alerts fire with probability sens for frauds, fpr otherwise
alert = np.where(fraud, rng.random(n) < sens, rng.random(n) < fpr)
# k = frauds among alerts, m = all alerts
k, m = int(fraud[alert].sum()), int(alert.sum())
# SciPy for the confidence interval
from scipy import stats
# 95% Wilson interval for the simulated posterior
ci = stats.binomtest(k, m).proportion_ci(0.95, method="wilson")
# Print the simulated posterior and the CI
print(f"Monte-Carlo (n={n:,}): {k/m:.4f}   95% Wilson CI [{ci.low:.4f}, {ci.high:.4f}]")
check("simulation agrees with the analytic posterior (analytic value inside the CI)", lambda: ci.low <= 0.153846 <= ci.high)

**Discuss.** (1) Which part of the formula is the *base rate*? (2) How would a **Naive Bayes** spam filter (Week 7) use exactly this rule with many words as evidence? (3) When should you prefer logic over probability for KR, and vice versa?

## 📝 Assignment (5 marks, individual, formative)
1. *(1)* Add **three new facts** and **two new queries** to the family knowledge base; show the answers.
2. *(2)* Translate three English sentences of your own into first-order logic and show the six clause-form steps for each.
3. *(1)* Prove (or fail to prove) a conclusion by resolution and print the proof.
4. *(1)* Repeat the fraud-alert Bayes calculation with different numbers and check it with the Monte-Carlo simulation (report the Wilson interval).

*Marked with the course rubric (class note, Table 0.3): correctness, evidence (numbers with CI and a named test), code clarity, interpretation. Submit the **executed notebook** (Runtime → Run all, outputs visible) as a Colab share link or `.ipynb` before the next lab.*

## Reflection
- What did *Skolemisation* do to $\exists y\,\forall x\,\text{Loves}(x,y)$ versus $\forall x\,\exists y\,\text{Loves}(x,y)$?
- Name one real KR challenge that neither logic nor probability solves cleanly (frame problem, common sense, scale…).

```text
Write your reflection here…
```

## Extensions
- Add **function symbols** (e.g. `Mother(x)`) and re-prove "every person has a grandmother".
- Replace breadth-first saturation by a **unit-preference** strategy and measure the number of clauses generated.
- Encode the Nixon diamond (Quaker ⇒ pacifist; Republican ⇒ ¬pacifist; Nixon is both). What does a *credulous* vs a *sceptical* default reasoner conclude?

In [ ]:
checks_summary()  # how many auto-checks have you passed?